In [ ]:
#%% read scraped data together
import pandas as pd
import json
import re
from tqdm import tqdm
from openai import OpenAI
import json.decoder
from os.path import join as j

pd.set_option('display.max_colwidth', 100)

# Initialize OpenAI client
client = OpenAI()

LETTER_PATH = "/Users/d.murray/Documents/science_criticism/data/raw/letter_content"

# Load CSV
df_pnas = pd.read_csv(j(LETTER_PATH, "pnas_criticals_scraped.csv"))
df_pnas = df_pnas[['doi','full_text']]
df_pnas['journal'] = "PNAS"

df_nature = pd.read_csv(j(LETTER_PATH, "nature_criticals_scraped.csv"))
df_nature = df_nature[df_nature.is_open_source==1][['doi','full_text']]
df_nature['journal'] = "Nature"

df_science = pd.read_csv(j(LETTER_PATH, "science_criticals_scraped.csv"))
df_science = df_science[df_science.is_open_source==1][['doi','full_text']]
df_science['journal'] = "Science"

df_prl = pd.read_csv(j(LETTER_PATH, "prl_criticals_scraped.csv"))
df_prl = df_prl[df_prl.is_open_source==1][['doi','full_text']]
df_prl['journal'] = "Physical Review Letters"

df_prabcde = pd.read_csv(j(LETTER_PATH, "prabcde_criticals_scraped.csv"))
df_prabcde = df_prabcde[df_prabcde.is_open_source==1][['doi','full_text','journal']]

df = pd.concat([df_pnas,df_nature,df_science,df_prl,df_prabcde],axis=0)


In [ ]:

#%% Get all GPT-analyzed results
# Output list for processed rows
output = []

# Iterate over rows
for idx, row in tqdm(df.iterrows(), total=len(df)):
    doi = row['doi']
    full_text = row['full_text']
    journal = row['journal']

    prompt = f"""
You are an academic expert analyzing scholarly critiques. The text that follows is content from a letter published in a major scientific journal addressing a paper previously published in the same venue.

Comment:
\"\"\"{full_text}\"\"\"

Based on this content, answer the following questions:
1. Is this content an instance of a critical letter? A critical letter is a document that issues explicit academic criticism against claims made in the targeted publication.
2. What is the level of interest of this critique for the general scientific audience? High interest raises uncertainty around the targeted paper's findings, likely owing to fundamental issue, and suggests that the critical letter should be cited alongside the target or that the target should not be cited at all. Low interest does not raise the uncertainty around a study and may not be of interest to the braoder scholarly community. If unsure, then label as low interest. Labels are one of: Low, High

Provide your answer in raw JSON only (no markdown, no extra formatting). Your output should be valid JSON like the following:

{{
  "is_critical_reason": "A brief explanation of whether or not this is a valid instance of a critical letter",
  "is_critical_bool": "Yes or No",
  "interest_reason": "brief exmaplanation of why you selected the level of interest that you did"
  "Interest": "The level of interest to the scientific community, one of: Low, High",
}}
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        reply = response.choices[0].message.content.strip()

        # Clean possible markdown formatting like ```json
        reply = re.sub(r"^```(?:json)?\s*|\s*```$", "", reply.strip(), flags=re.IGNORECASE).strip()

        # Parse JSON
        parsed = json.loads(reply)
        parsed['doi'] = doi
        parsed['full_text'] = full_text
        parsed['journal'] = journal
        output.append(parsed)

    except json.decoder.JSONDecodeError as e:
        print(f"JSON decode error at DOI {doi}: {e}")
        output.append({
            "doi": doi,
            "full_text": full_text,
            "journal": journal,
            "is_critical_bool": "error",
            "is_critical_reason": "error",
            "new_evidence_bool": "error",
            "new_evidence_reason": "error",
            "critique_type": "error",
            "critique_type_reason": "error"
        })

    except Exception as e:
        print(f"Other error at DOI {doi}: {e}")
        output.append({
            "doi": doi,
            "full_text": full_text,
            "journal": journal,
            "is_critical_bool": "error",
            "is_critical_reason": "error",
            "new_evidence_bool": "error",
            "new_evidence_reason": "error",
            "critique_type": "error",
            "critique_type_reason": "error"
        })

#%%
# Save to CSV
results_df = pd.DataFrame(output)
#results_df.to_csv("../data/critical_letters_text/all_critique_analysis.csv", index=False, header=True)
#results_df.head()
# %%
#results_df.value_counts('critique_type')
# %%
#results_df.value_counts('is_critical_bool')
#%%
#results_df.value_counts('new_evidence_bool')
# %%


In [ ]:
results_df.to_csv("~/Desktop/letter_text_analysis_data.csv")

In [ ]:
results_df